# OISST model comparison

Select a GPU runtime and Run all. Code is installed from the latest main
of chahineNejm/Generalized-Patchwork; the resolved commit is saved.
Defaults live in experiments/default_hyperparameters.py. Edit overrides
below, not model definitions. No model or dataset implementation is embedded.


## 1. Load code and dependencies


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
import hashlib, importlib, json, os, subprocess, sys
from pathlib import Path

REPOSITORY = 'https://github.com/chahineNejm/Generalized-Patchwork.git'
BRANCH = 'main'
ROOT = Path('/content/gpw_oisst')
if ROOT.exists():
    raise FileExistsError("Checkout already exists; use a fresh Colab runtime")
if any(name == "pwgp" or name.startswith("pwgp.") or name == "experiments"
       or name.startswith("experiments.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading a new source version")
os.environ["GIT_TERMINAL_PROMPT"] = "0"
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, "--single-branch",
                REPOSITORY, str(ROOT)], check=True)
subprocess.run(["git", "-C", str(ROOT), "sparse-checkout", "set", "--no-cone",
                "/src/", "/experiments/", "/pyproject.toml", "/README.md"], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", BRANCH], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip()
SOURCE_ZIP = subprocess.check_output(["git", "-C", str(ROOT), "archive", "--format=zip",
                                      COMMIT, "pyproject.toml", "README.md", "src", "experiments"])
SOURCE_PROVENANCE = {"repository": REPOSITORY, "branch": BRANCH, "commit": COMMIT,
                     "source_sha256": hashlib.sha256(SOURCE_ZIP).hexdigest()}
os.environ["MPLCONFIGDIR"] = str(ROOT / "results/.matplotlib")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.chdir(ROOT)
sys.path[:0] = [str(ROOT / "src"), str(ROOT)]
importlib.invalidate_caches()
_GPW_CELL_RESULT = print(SOURCE_PROVENANCE)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
_GPW_CELL_RESULT = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "--editable", str(ROOT) + "[data,ukhousing]"], check=True)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 2. Defaults and explicit overrides


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
from dataclasses import replace
from datetime import UTC, datetime
from IPython.display import Image, display
from google.colab import files
import pandas as pd
import torch
from experiments.default_hyperparameters import oisst_defaults
from experiments.oisst_benchmark import run_comparison
from experiments.run_selection import select_runs

DATA, RUNS = oisst_defaults(device="cuda:0")
MODELS = ["SVGP", "GPW", "ProSpar", "VNNGP", "Standard Patchwork"]
# Optional overrides, e.g. {"GPW": {"epochs": 150, "warmup_epochs": 30}}.
OVERRIDES = {
    "SVGP": {"epochs": 2, "warmup_epochs": 1},
    "GPW": {"epochs": 2, "warmup_epochs": 1},
    "ProSpar": {"epochs": 2, "warmup_epochs": 1},
    "VNNGP": {"epochs": 2, "warmup_epochs": 1},
    "Standard Patchwork": {"epochs": 2},
}
DATA_OVERRIDES = {}  # e.g. {"save_inducing_positions": False}
RUNS = select_runs(RUNS, models=MODELS, overrides=OVERRIDES)
DATA = replace(DATA, **DATA_OVERRIDES)
RUN_ROOT = ROOT / "results/oisst" / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; there is no CPU fallback")
display(pd.DataFrame([{"run": name, **config.__dict__} for name, config in RUNS]))
_GPW_CELL_RESULT = print(DATA)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 3. Run, visualize and save


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
live = display("Starting OISST comparison...", display_id=True)

def show_progress(path, epoch):
    live.update(Image(filename=str(path / "visuals/progress.png")))
    display(Image(filename=str(path / "visuals/training.png")))

def show_completed(path, archive, metrics):
    display(pd.DataFrame([metrics]).T)
    display(Image(filename=str(path / "visuals/maps.png")))
    display(Image(filename=str(path / "visuals/metrics.png")))
    display(Image(filename=str(path / "visuals/hyperparameters.png")))
    metadata = json.loads((path / "model_metadata.json").read_text())
    if metadata["inducing_positions_saved"]:
        display(Image(filename=str(path / "visuals/inducing_positions.png")))
    files.download(str(archive))

result = run_comparison(
    DATA, RUNS, directory=RUN_ROOT, data_root=ROOT / "data",
    source_provenance=SOURCE_PROVENANCE, source_zip=SOURCE_ZIP,
    on_snapshot=show_progress, on_complete=show_completed,
)
display(pd.read_json(result / "comparison.json"))
_GPW_CELL_RESULT = print("Results:", result)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT


## 4. Download the complete comparison


In [ ]:
if "_GPW_STAGE_FAILED" not in globals():
    _GPW_STAGE_FAILED = False
if _GPW_STAGE_FAILED:
    raise RuntimeError("An earlier cell failed; restart the runtime before continuing")
_GPW_STAGE_FAILED = True
import shutil
archive = shutil.make_archive(str(RUN_ROOT), "zip", root_dir=RUN_ROOT)
_GPW_CELL_RESULT = files.download(archive)
_GPW_STAGE_FAILED = False
_GPW_CELL_RESULT
